### What is Delta Lake format?

**Delta Lake is a storage format/table format built on top of Parquet** that adds database-like features to data stored in a data lake.

Think of it like this:
> **Parquet = stores data efficiently**   
> **Delta Lake = Parquet + transaction/history/reliability features**


### 1. Without Delta Lake

Suppose you have data in S3:

```text
s3://company-data/customers/
    part-0001.parquet
    part-0002.parquet
    part-0003.parquet
```

These are just Parquet files.

If you update a customer's record, there isn't a built-in transaction mechanism coordinating all those files.

---

### 2. With Delta Lake

A Delta table looks roughly like:

```text
s3://company-data/customers/

    part-0001.parquet
    part-0002.parquet
    part-0003.parquet

    _delta_log/
        00000000000000000000.json
        00000000000000000001.json
        00000000000000000002.json
```

The important part is:

```text
_delta_log/
```

This is the **transaction log**.

It keeps track of things such as:

* Which Parquet files belong to the table
* Which files were added
* Which files were removed
* Table schema
* Table versions
* Changes made by operations such as `MERGE`, `UPDATE`, and `DELETE`

So Delta Lake knows the **state of the table at each version**.

---

## Why do we use Delta Lake?

The major advantage is that it provides **data-lake + database capabilities**.

| Feature               | Parquet   | Delta Lake |
| --------------------- | --------- | ---------- |
| Columnar storage      | ✅         | ✅          |
| Compression           | ✅         | ✅          |
| Schema enforcement    | Limited   | ✅          |
| ACID transactions     | ❌         | ✅          |
| `UPDATE`              | Difficult | ✅          |
| `DELETE`              | Difficult | ✅          |
| `MERGE` / Upsert      | Difficult | ✅          |
| Time Travel           | ❌         | ✅          |
| Transaction history   | ❌         | ✅          |
| Concurrent operations | Limited   | Better     |


### Interview answer

If an interviewer asks **"What is Delta Lake?"**, you can say:

> **Delta Lake is an open-source storage/table format that sits on top of Parquet and provides ACID transactions, schema enforcement, time travel, and operations like MERGE, UPDATE, and DELETE. It uses a transaction log called `_delta_log` to maintain the consistent state and history of the table.**

For a **Data Engineer interview**, the next important topic after this is **how `_delta_log` actually works during INSERT, UPDATE and MERGE**.


In [66]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StringType, StructField, IntegerType
from pyspark.sql.functions import col, upper, dense_rank
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip
from delta import *

builder = (
    SparkSession.Builder()
    .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .appName("DeltaTest")
    .master("local[*]")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [67]:
# creating schema 
# id,name,age,salary,country,dept
emp_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("age", IntegerType(), False),
    StructField("salary", IntegerType(), True),
    StructField("country", StringType(), True),
    StructField("dept", StringType(), True)
])

# reading file
empDf = spark.read.format("csv")\
    .option("header", True)\
    .schema(emp_schema)\
    .load("data/emp_data_merge.csv")

In [68]:
empDf.show(50000, False)

+---+-------+---+------+-------+-----------+
|id |name   |age|salary|country|dept       |
+---+-------+---+------+-------+-----------+
|1  |manish |26 |20000 |india  |IT         |
|2  |rahul  |30 |40000 |germany|engineering|
|3  |pawan  |12 |60000 |india  |sales      |
|4  |roshini|44 |30000 |uk     |engineering|
|5  |raushan|35 |70000 |india  |sales      |
|6  |Vishva |29 |200000|uk     |IT         |
|7  |adam   |37 |65000 |us     |IT         |
|8  |chris  |16 |40000 |us     |sales      |
|7  |adam   |37 |65000 |us     |IT         |
+---+-------+---+------+-------+-----------+



### Wrting the data from df to delta table

In [69]:
# write df as delta table
empDf.write.format("delta").mode("overwrite").saveAsTable("employee_delta_tbl")


### reading the file from delta table

In [70]:
emp = spark.read.format("delta").table("employee_delta_tbl")
emp.show()
emp.printSchema()

+---+-------+---+------+-------+-----------+
| id|   name|age|salary|country|       dept|
+---+-------+---+------+-------+-----------+
|  1| manish| 26| 20000|  india|         IT|
|  2|  rahul| 30| 40000|germany|engineering|
|  3|  pawan| 12| 60000|  india|      sales|
|  4|roshini| 44| 30000|     uk|engineering|
|  5|raushan| 35| 70000|  india|      sales|
|  6| Vishva| 29|200000|     uk|         IT|
|  7|   adam| 37| 65000|     us|         IT|
|  8|  chris| 16| 40000|     us|      sales|
|  7|   adam| 37| 65000|     us|         IT|
+---+-------+---+------+-------+-----------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- dept: string (nullable = true)



#### Applying transformation on top 2 higher salary by 30 % increment

In [71]:
# select most 2 emp salary
emp_selected = emp\
    .select(col("*"),
    dense_rank()\
        .over(Window.orderBy(emp.salary.desc()).rowsBetween(Window.unboundedPreceding, Window.currentRow))\
        .alias("rank"))\
    .filter(col("rank")<3)

# remove ranking column
emp_selected=emp_selected.drop(col("rank"))
emp_selected.show()

# update salary by 30%
emp_selected = emp_selected.withColumn("salary", (emp_selected.salary*1.3).cast("int"))
emp_selected.show()

+---+-------+---+------+-------+-----+
| id|   name|age|salary|country| dept|
+---+-------+---+------+-------+-----+
|  6| Vishva| 29|200000|     uk|   IT|
|  5|raushan| 35| 70000|  india|sales|
+---+-------+---+------+-------+-----+



26/09/23 06:47:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 0

+---+-------+---+------+-------+-----+
| id|   name|age|salary|country| dept|
+---+-------+---+------+-------+-----+
|  6| Vishva| 29|260000|     uk|   IT|
|  5|raushan| 35| 91000|  india|sales|
+---+-------+---+------+-------+-----+



### Merging the data using delta tables

In [72]:
from delta.tables import DeltaTable

delta_emp_table = DeltaTable.forName(spark, "employee_delta_tbl")

delta_emp_table.alias("target")\
    .merge(
        emp_selected.alias("source"),
        "target.id = source.id"
    )\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

26/09/23 06:47:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:10 WARN MapPartitionsRDD: RDD 942 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [73]:
# validating the above merge
spark.sql("Select * from employee_delta_tbl").show()

+---+-------+---+------+-------+-----------+
| id|   name|age|salary|country|       dept|
+---+-------+---+------+-------+-----------+
|  1| manish| 26| 20000|  india|         IT|
|  2|  rahul| 30| 40000|germany|engineering|
|  3|  pawan| 12| 60000|  india|      sales|
|  4|roshini| 44| 30000|     uk|engineering|
|  5|raushan| 35| 91000|  india|      sales|
|  6| Vishva| 29|260000|     uk|         IT|
|  7|   adam| 37| 65000|     us|         IT|
|  7|   adam| 37| 65000|     us|         IT|
|  8|  chris| 16| 40000|     us|      sales|
+---+-------+---+------+-------+-----------+



#### we can do merge using spark sql also

In [74]:
# creating the temp view using the emp df 
emp_selected = emp_selected.withColumn("salary", (emp_selected.salary*2).cast("int"))
emp_selected.createOrReplaceTempView("emp_selected_view")

# Applying the merge on employee_delta_tbl
spark.sql(
    """
merge into employee_delta_tbl as target
using emp_selected_view as source
on target.id = source.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
"""
)

26/09/23 06:47:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/23 06:47:13 WARN MapPartitionsRDD: RDD 1012 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [76]:
# validating the record
spark.sql("select * from employee_delta_tbl").show()

+---+-------+---+------+-------+-----------+
| id|   name|age|salary|country|       dept|
+---+-------+---+------+-------+-----------+
|  1| manish| 26| 20000|  india|         IT|
|  2|  rahul| 30| 40000|germany|engineering|
|  3|  pawan| 12| 60000|  india|      sales|
|  4|roshini| 44| 30000|     uk|engineering|
|  5|raushan| 35|236600|  india|      sales|
|  6| Vishva| 29|676000|     uk|         IT|
|  7|   adam| 37| 65000|     us|         IT|
|  7|   adam| 37| 65000|     us|         IT|
|  8|  chris| 16| 40000|     us|      sales|
+---+-------+---+------+-------+-----------+



In [81]:
# checking spark history
spark.sql("describe history employee_delta_tbl").show(truncate=False)

# going to previous version
spark.sql("select * from employee_delta_tbl version as of 0").show()

+-------+-----------------------+------+--------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+

In [75]:
# cleaning up the spark-warehouse folder
# ! rm -r spark-warehouse/employee_delta_tbl